# registry

> Local discovery of running paar servers. Each `serve()` process writes a tiny
> JSON entry here so any frontend (web or `paar-tui`) can list live environments
> and switch between them — identified by repo/package name.

One JSON file per live server under `~/.paar/reg` (override: `$PAAR_REG_DIR`), pruned by
pid-liveness on read. This is also the discovery root agents use: `paar-mcp --env NAME`
resolves its target here on every call. Owner-token files (`token-<port>`) live alongside
the entries — written by `serve()`/`inspector()`, read by `paar-tui`, never listed in the
registry JSON itself.

In [ ]:
#| default_exp registry

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, json, socket, subprocess, time
from pathlib import Path

def reg_dir():
    "Directory holding one JSON file per live paar server (override with $PAAR_REG_DIR)."
    d = Path(os.environ.get('PAAR_REG_DIR', Path.home()/'.paar'/'reg'))
    d.mkdir(parents=True, exist_ok=True); return d

def _git_name(cwd=None):
    "Basename of the enclosing git repo, or None."
    try:
        r = subprocess.run(['git','-C',str(cwd or Path.cwd()),'rev-parse','--show-toplevel'],
                           capture_output=True, text=True, timeout=2)
        if r.returncode==0 and r.stdout.strip(): return Path(r.stdout.strip()).name
    except Exception: pass
    return None

def _pkg_name(ns):
    "Top-level package name inferred from a namespace's __package__/__name__, or None."
    if not ns: return None
    for k in ('__package__','__name__'):
        v = ns.get(k)
        if v and v!='__main__': return str(v).split('.')[0]
    return None

def env_name(explicit=None, ns=None):
    "Env label: explicit > git repo name > package name > cwd basename."
    return explicit or _git_name() or _pkg_name(ns) or Path.cwd().name

def free_port(start=8000, tries=50):
    "First bindable TCP port at/after start."
    for p in range(start, start+tries):
        with socket.socket() as s:
            try: s.bind(('127.0.0.1', p)); return p
            except OSError: continue
    raise OSError(f'no free port in {start}..{start+tries}')

In [ ]:
#| export
def _alive(pid):
    "Is a process with pid running and not a zombie? (os.kill(pid,0) also succeeds for a defunct process)."
    try: os.kill(pid, 0)
    except ProcessLookupError: return False
    except PermissionError: return True
    except Exception: return False
    try:
        st = subprocess.run(['ps','-o','state=','-p',str(pid)], capture_output=True, text=True, timeout=2).stdout.strip()
        if st[:1] == 'Z': return False   # zombie/defunct: reads as alive but is dead
    except Exception: pass
    return True

def register(name, port, base=None):
    "Write this server's registry entry and return it."
    e = dict(name=name, port=port, pid=os.getpid(), cwd=str(Path.cwd()),
             base=base or f'http://127.0.0.1:{port}', started=time.time())
    (reg_dir()/f'{port}.json').write_text(json.dumps(e)); return e

def unregister(port):
    "Remove this server's registry entry."
    try: (reg_dir()/f'{port}.json').unlink()
    except FileNotFoundError: pass

def active():
    "Live registry entries, pruning any whose process has died."
    out = []
    for f in sorted(reg_dir().glob('*.json')):
        try: e = json.loads(f.read_text())
        except Exception: continue
        if _alive(e.get('pid', -1)): out.append(e)
        else:
            try: f.unlink()
            except Exception: pass
    return out

In [ ]:
import tempfile
os.environ['PAAR_REG_DIR'] = tempfile.mkdtemp()

def test_register_active_roundtrip():
    unregister(9999)
    e = register('demo', 9999)
    assert e['name']=='demo' and e['pid']==os.getpid() and e['base'].endswith(':9999')
    assert 'demo' in [x['name'] for x in active()]
    unregister(9999)
    assert 9999 not in [x['port'] for x in active()]

def test_active_prunes_dead():
    p = reg_dir()/'8123.json'
    p.write_text(json.dumps(dict(name='ghost', port=8123, pid=2147480000, cwd='/', base='x', started=0)))
    assert 'ghost' not in [x['name'] for x in active()]   # dead pid pruned
    assert not p.exists()

def test_env_name_precedence():
    assert env_name('myenv')=='myenv'                     # explicit wins
    assert env_name(ns={'__name__':'pkg.sub'})            # falls to git repo name here (in-repo)
    assert env_name()                                     # non-empty (git/pkg/cwd)

def test_free_port_bindable():
    p = free_port(8000)
    with socket.socket() as s: s.bind(('127.0.0.1', p))   # returned port is actually free

for t in [test_register_active_roundtrip, test_active_prunes_dead,
          test_env_name_precedence, test_free_port_bindable]: t()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()